# 🏛️ Sarthika AGI Core v1 — Full Cloud Training & GGUF Export Pipeline
### Deliberative System 2 Reasoning, Neuro-Symbolic Verification, and CPU-Optimized GGUF Packaging

> **Kaggle Hardware Setup (Essential):**
> 1. In the right-hand panel under **Settings**:
>    * **Accelerator**: Select **GPU T4 x2** (or **GPU P100**).
>    * **Internet**: Turn **Internet ON** (Required to download foundation weights and tools).
> 2. Click **Run All** (or run cells sequentially from Step 1 to Step 8).
> 3. Once complete, your personal model `sarthika_agi_v1_q4_k_m.gguf` will be saved in `/kaggle/working/` ready to download!


In [1]:
# Step 1: Verify Hardware Acceleration
import os
# Restrict this process to a single physical GPU. This MUST happen before
# torch is imported/initializes CUDA anywhere in the kernel -- Kaggle's
# "GPU T4 x2" setting exposes 2 GPUs, and HuggingFace Trainer wraps the
# model in legacy torch.nn.DataParallel whenever torch.cuda.device_count()
# > 1, regardless of where the model's weights were actually placed. That
# DataParallel wrap is what caused the "chunk expects at least a
# 1-dimensional tensor" crash in Step 6 (it tries to scatter a scalar loss
# tensor across 2 GPUs). Restricting to 1 GPU here removes the problem at
# its root, rather than trying to patch Trainer's internals after the fact.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import psutil

print("=" * 65)
print("🔍 AUDITING KAGGLE HARDWARE ACCELERATION")
print("=" * 65)

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU is NOT detected! Please go to Notebook Settings -> Accelerator -> select GPU T4 x2 or P100.")

gpu_name = torch.cuda.get_device_name(0)
gpu_count = torch.cuda.device_count()
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"✅ GPU Detected     : {gpu_name} (Visible to this process: {gpu_count})")
print(f"✅ Available VRAM   : {vram_gb:.2f} GB")
print(f"✅ Host System RAM  : {ram_gb:.2f} GB")
print(f"✅ PyTorch Version  : {torch.__version__} | CUDA Version: {torch.version.cuda}")
print("ℹ️  Running single-GPU by design (avoids Trainer's legacy DataParallel bug).")
print("=" * 65)


🔍 AUDITING KAGGLE HARDWARE ACCELERATION
✅ GPU Detected     : Tesla T4 (Visible to this process: 1)
✅ Available VRAM   : 14.56 GB
✅ Host System RAM  : 31.35 GB
✅ PyTorch Version  : 2.10.0+cu128 | CUDA Version: 12.8
ℹ️  Running single-GPU by design (avoids Trainer's legacy DataParallel bug).


In [2]:
# Step 2: Install ALL Fine-Tuning Libraries (single pinned install) & Compile GGUF Engine
import subprocess
import sys
import os

print("Installing pinned, mutually-compatible dependencies for the entire pipeline...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==1.1.1",
    "peft==0.13.2",
    "trl==0.11.4",
    "datasets==3.1.0",
    "scipy",
    "requests>=2.31.0",
    "sentencepiece",
    "protobuf",
    "tiktoken",
    "gguf>=0.1.0"
], check=True)

# bitsandbytes is deliberately NOT pinned. Unlike the pure-Python libraries
# above, it ships precompiled CUDA binaries that must match the CUDA toolkit
# actually present on the host. Kaggle's base image CUDA version changes
# over time (currently CUDA 12.x), so pinning an old bitsandbytes release
# risks it silently falling back to a CPU-only binary with no GPU NF4
# kernels — which crashes later with "undefined symbol" errors instead of
# failing at install time. Always install the latest release instead.
print("Installing latest bitsandbytes (must match host CUDA toolkit, so left unpinned)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)

# Remove Triton: not used (torch.compile is disabled in this pipeline), and on
# Kaggle's preinstalled torch build, Triton 3.x dropped the legacy 'triton.ops'
# module that bitsandbytes' optional kernel path imports unconditionally.
# Uninstalling makes bitsandbytes take its safe, well-tested CUDA-kernel path.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "triton"], check=False)

# Sanity check: confirm bitsandbytes actually found a GPU binary before we
# spend time downloading the 15GB model, rather than failing deep inside
# model loading with an unclear error.
check = subprocess.run(
    [sys.executable, "-c",
     "import bitsandbytes as bnb; print('bitsandbytes', bnb.__version__, 'GPU binary OK')"],
    capture_output=True, text=True
)
print(check.stdout.strip())
if check.returncode != 0 or "GPU binary OK" not in check.stdout:
    print("⚠️ bitsandbytes GPU binary check failed — see stderr below:")
    print(check.stderr[-2000:])

print("Compiling llama.cpp GGUF converter & quantization engine...")
if not os.path.exists("llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git"], check=True)
    subprocess.run(["cmake", "-B", "llama.cpp/build", "-S", "llama.cpp", "-DGGML_CUDA=OFF"], check=True)
    subprocess.run(["cmake", "--build", "llama.cpp/build", "--config", "Release", "-j4"], check=True)

print("✅ All dependencies installed once, pinned where it matters, and consistent for the rest of the notebook.")


Installing pinned, mutually-compatible dependencies for the entire pipeline...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.


Installing latest bitsandbytes (must match host CUDA toolkit, so left unpinned)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.0 MB/s eta 0:00:00
bitsandbytes 0.50.2 GPU binary OK
Compiling llama.cpp GGUF converter & quantization engine...


Cloning into 'llama.cpp'...


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.5.0-dev
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD


CMAKE_BUILD_TYPE=Release


-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.25.3
-- ggml commit:  9adc7f4
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.2")
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (1.8s)
-- Generating done (0.3s)
-- Build files have been written to: /kaggle/working/llama.cpp/build
[  1%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/h

In [3]:
# Step 3: Construct the Sarthika Deliberative Reasoning Corpus (AGI Protocol)
import json
from datasets import Dataset

print("Generating high-density Sarthika System 2 deliberative training dataset...")

SYSTEM_PROMPT = """You are Sarthika AGI, an autonomous deliberative reasoning engine.
When given an objective, you follow the Sarthika Cognitive Protocol:
1. <thought>: Metacognitively decompose the objective, state invariants, edge cases, and algorithmic strategy.
2. <code>: Synthesize self-contained, typed, verified Python code that computes results dynamically.
3. <verification>: Programmatically evaluate execution output, verify invariants, and diagnose errors.
4. <conclusion>: Formulate a precise, mathematically and computationally sound answer."""

raw_trajectories = [
    # Trajectory 1: Euler Totient & Gauss Primitive Root Theorem
    {
        "prompt": "Compute the Euler Totient function phi(N) for N = 120, find its prime factors, and determine if a primitive root modulo 120 exists based on the Gauss Primitive Root Theorem.",
        "thought": """1. Metacognitive Decomposition: Need to compute phi(120), factorize phi(120), and evaluate Gauss Primitive Root Theorem for modulus 120.
2. Invariant Analysis: Gauss Theorem states a primitive root modulo N exists if and only if N is 2, 4, p^k, or 2*p^k for an odd prime p.
3. Algorithmic Plan: N=120 = 2^3 * 3 * 5. Since N has multiple odd prime factors and 2^3 (k>=3), no primitive root can exist. Programmatically compute phi(120) = 120 * (1-1/2) * (1-1/3) * (1-1/5) = 32.""",
        "code": """def euler_totient(n: int) -> int:
    result = n
    p = 2
    temp = n
    while p * p <= temp:
        if temp % p == 0:
            while temp % p == 0:
                temp //= p
            result -= result // p
        p += 1
    if temp > 1:
        result -= result // temp
    return result

def prime_factors(n: int) -> list:
    factors = []
    d = 2
    temp = n
    while d * d <= temp:
        if temp % d == 0:
            factors.append(d)
            while temp % d == 0:
                temp //= d
        d += 1
    if temp > 1:
        factors.append(temp)
    return factors

def has_primitive_root(n: int) -> tuple[bool, str]:
    if n in (2, 4):
        return True, f"{n} has primitive roots (base cases 2, 4)."
    if n % 2 == 0:
        temp = n // 2
        if temp % 2 == 0:
            return False, f"{n} is divisible by 4 and > 4; by Gauss Theorem, no primitive root exists."
    else:
        temp = n
    factors = prime_factors(temp)
    if len(factors) == 1:
        return True, f"{n} is of the form p^k or 2*p^k for odd prime {factors[0]}."
    return False, f"{n} has multiple distinct odd prime factors {factors}; by Gauss Theorem, no primitive root exists."

phi_120 = euler_totient(120)
factors_phi = prime_factors(phi_120)
has_root, justification = has_primitive_root(120)

print(f"phi(120) = {phi_120}")
print(f"Prime factors of phi(120): {factors_phi}")
print(f"Primitive root existence: {has_root} -> {justification}")
""",
        "verification": "Execution verified: phi(120) evaluated correctly to 32. Prime factors of 32 are [2]. Since 120 = 8 * 15, it contains 2^3 and two distinct odd primes (3 and 5), strictly violating Gauss's condition.",
        "conclusion": "1. The Euler Totient value phi(120) is 32.\n2. The unique prime factor of phi(120) = 32 is {2}.\n3. By the Gauss Primitive Root Theorem, a primitive root modulo 120 DOES NOT exist because 120 is divisible by 8 and has multiple odd prime factors (3 and 5)."
    },

    # Trajectory 2: Lucas-Lehmer Mersenne Primality Verification
    {
        "prompt": "Verify whether 2^31 - 1 is prime using the Lucas-Lehmer test sequence and check its digits.",
        "thought": """1. Objective: Determine primality of Mersenne number M_31 = 2^31 - 1 using Lucas-Lehmer recurrence.
2. Invariant Check: Lucas-Lehmer recurrence starts with S_0 = 4, S_i = (S_{i-1}^2 - 2) mod M_p. M_p is prime iff S_{p-2} == 0.
3. Scalability: For p=31, p-2 = 29 iterations. M_31 = 2147483647 (10 digits).""",
        "code": """def lucas_lehmer_test(p: int) -> tuple[bool, int]:
    if p == 2:
        return True, 3
    m_p = (1 << p) - 1
    s = 4
    for _ in range(p - 2):
        s = (s * s - 2) % m_p
    return s == 0, m_p

p = 31
is_prime, mersenne_val = lucas_lehmer_test(p)
digits = str(mersenne_val)

print(f"M_{p} = {mersenne_val}")
print(f"Digit count: {len(digits)}")
print(f"First 3 digits: {digits[:3]}, Last 3 digits: {digits[-3:]}")
print(f"Is M_{p} prime via Lucas-Lehmer: {is_prime}")
""",
        "verification": "Lucas-Lehmer sequence for p=31 iterated 29 steps. S_29 mod (2^31 - 1) evaluated strictly to 0. Number of digits is 10.",
        "conclusion": "2^31 - 1 = 2,147,483,647 is confirmed to be a prime number (the 8th Mersenne prime). It contains 10 decimal digits, starting with '214' and ending with '647'."
    },

    # Trajectory 3: O(log N) Fibonacci Matrix Exponentiation & Factorization
    {
        "prompt": "Using matrix exponentiation, compute the 30th Fibonacci number F_30, factorize it completely, and verify if any of its prime factors divide 2^15 - 1.",
        "thought": """1. Objective: Compute F(30) in O(log N) time using 2x2 matrix [[1, 1], [1, 0]], find its prime factorization, and test divisibility against 2^15 - 1 = 32767.
2. Matrix Doubling Invariant: [[1,1],[1,0]]^n = [[F_{n+1}, F_n], [F_n, F_{n-1}]].
3. Verification: F(30) = 832040. Factors: 832040 = 2^3 * 5 * 11 * 31 * 61.""",
        "code": """def matrix_multiply_2x2(A, B):
    return [
        [A[0][0]*B[0][0] + A[0][1]*B[1][0], A[0][0]*B[0][1] + A[0][1]*B[1][1]],
        [A[1][0]*B[0][0] + A[1][1]*B[1][0], A[1][0]*B[0][1] + A[1][1]*B[1][1]]
    ]

def matrix_power_2x2(M, power):
    result = [[1, 0], [0, 1]]
    base = M
    while power > 0:
        if power % 2 == 1:
            result = matrix_multiply_2x2(result, base)
        base = matrix_multiply_2x2(base, base)
        power //= 2
    return result

def fibonacci_matrix(n: int) -> int:
    if n <= 0: return 0
    if n == 1: return 1
    T = [[1, 1], [1, 0]]
    Tn = matrix_power_2x2(T, n - 1)
    return Tn[0][0]

def prime_factor_dict(n: int) -> dict:
    factors = {}
    d = 2
    temp = n
    while d * d <= temp:
        while temp % d == 0:
            factors[d] = factors.get(d, 0) + 1
            temp //= d
        d += 1
    if temp > 1:
        factors[temp] = factors.get(temp, 0) + 1
    return factors

f_30 = fibonacci_matrix(30)
factors = prime_factor_dict(f_30)
target = (1 << 15) - 1

divisible_factors = [p for p in factors if target % p == 0]

print(f"F_30 = {f_30}")
print(f"Prime Factorization of F_30: {factors}")
print(f"Target 2^15 - 1 = {target}")
print(f"Factors dividing target: {divisible_factors}")
""",
        "verification": "Matrix power executed in log2(29) multiplications. F_30 = 832040. Prime factors are {2: 3, 5: 1, 11: 1, 31: 1, 61: 1}. Testing 32767: 32767 = 7 * 31 * 151. Prime factor 31 strictly divides 2^15 - 1 (32767 / 31 = 1057).",
        "conclusion": "1. The 30th Fibonacci number F_30 is 832,040.\n2. Its complete prime factorization is 2^3 * 5 * 11 * 31 * 61.\n3. The prime factor 31 divides 2^15 - 1 (since 2^15 - 1 = 32,767 = 31 * 1,057)."
    },

    # Trajectory 4: Reflexion & Bug Diagnosis Trace
    {
        "prompt": "Construct a function to compute modular inverse using the Extended Euclidean Algorithm, handling non-coprime exceptions safely.",
        "thought": """1. Objective: Implement extended_gcd(a, m) returning (g, x, y) such that a*x + m*y = gcd(a, m).
2. Error Preemption: If gcd(a, m) != 1, modular inverse does not exist; code must raise ValueError or return None rather than returning unverified values.
3. Invariant: If inverse exists, (a * inv) % m == 1.""",
        "code": """def extended_gcd(a: int, b: int) -> tuple[int, int, int]:
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd, x, y

def mod_inverse(a: int, m: int) -> int:
    gcd, x, _ = extended_gcd(a % m, m)
    if gcd != 1:
        raise ValueError(f"Modular inverse does not exist: gcd({a}, {m}) = {gcd} != 1")
    return (x % m + m) % m

test_cases = [(3, 11), (10, 17), (6, 9)]
for a, m in test_cases:
    try:
        inv = mod_inverse(a, m)
        assert (a * inv) % m == 1
        print(f"inv({a} mod {m}) = {inv} -> Verified: ({a} * {inv}) % {m} = {(a*inv)%m}")
    except ValueError as e:
        print(f"Handled expected exception for ({a}, {m}): {e}")
""",
        "verification": "Extended Euclidean algorithm correctly computed modular inverses: 3 mod 11 -> 4, 10 mod 17 -> 12. For non-coprime pair (6, 9), correctly raised ValueError without crashing.",
        "conclusion": "The modular inverse engine is mathematically sound. Inverses for coprime pairs are verified, and singular/non-coprime exceptions are handled gracefully."
    },

    # Trajectory 5: Directed Acyclic Graph (DAG) Dynamic Pipeline Synthesis
    {
        "prompt": "Design a Directed Acyclic Graph (DAG) pipeline orchestrator that resolves task dependencies, detects circular references, and executes functions in topological order.",
        "thought": """1. Objective: Implement a topological dependency scheduler using Kahn's algorithm (in-degree tracking).
2. Causal Invariant: If the graph has cycles, topological sort must detect remaining non-zero in-degrees and raise a CycleError.
3. Execution: Pass verified output states sequentially through dependencies.""",
        "code": """from collections import deque
from typing import Dict, List, Callable, Any

class DAGOrchestrator:
    def __init__(self):
        self.tasks: Dict[str, Callable] = {}
        self.dependencies: Dict[str, List[str]] = {}

    def add_task(self, name: str, func: Callable, deps: List[str] = None):
        self.tasks[name] = func
        self.dependencies[name] = deps or []

    def execute(self) -> Dict[str, Any]:
        in_degree = {name: 0 for name in self.tasks}
        graph = {name: [] for name in self.tasks}

        for task, deps in self.dependencies.items():
            for dep in deps:
                if dep not in self.tasks:
                    raise KeyError(f"Missing dependency: {dep} required by {task}")
                graph[dep].append(task)
                in_degree[task] += 1

        queue = deque([t for t, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            node = queue.popleft()
            order.append(node)
            for neighbor in graph[node]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)

        if len(order) != len(self.tasks):
            raise ValueError("Cycle detected in skill graph! Pipeline execution aborted.")

        context = {}
        for task in order:
            context[task] = self.tasks[task](context)
        return context

dag = DAGOrchestrator()
dag.add_task("seed", lambda ctx: 10)
dag.add_task("square", lambda ctx: ctx["seed"] ** 2, deps=["seed"])
dag.add_task("add_five", lambda ctx: ctx["square"] + 5, deps=["square"])

results = dag.execute()
print("DAG Execution Trace:", results)
""",
        "verification": "Topological ordering correctly resolved: ['seed', 'square', 'add_five']. Computed final state: 105. Cycle detection validated via Kahn's algorithm in-degree check.",
        "conclusion": "The DAG orchestrator successfully executes dependency-ordered pipelines and enforces acyclic causal execution."
    }
]

formatted_data = []
for item in raw_trajectories:
    assistant_content = f"""<thought>
{item['thought']}
</thought>

<code>
{item['code']}
</code>

<verification>
{item['verification']}
</verification>

<conclusion>
{item['conclusion']}
</conclusion>"""

    dialogue = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": assistant_content}
    ]
    formatted_data.append({"messages": dialogue})

dataset = Dataset.from_list(formatted_data)
print(f"Generated dataset containing {len(dataset)} rich deliberative cognitive trajectories.")


Generating high-density Sarthika System 2 deliberative training dataset...
Generated dataset containing 5 rich deliberative cognitive trajectories.


In [4]:
# Step 4: Load Qwen2.5-Coder Foundation Engine in 4-bit NF4 Precision
import os
import torch

# Disable TorchDynamo / JIT compilation via official env flags (no triton needed).
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["ACCELERATE_DISABLE_TORCH_COMPILE"] = "1"

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers
print(f"Verified Active Transformers Version: {transformers.__version__}")

# MODEL_ID was Qwen/Qwen2.5-Coder-7B-Instruct. Switched to the 3B variant:
# this Kaggle session has a hard ~19.5GB disk quota. Any GGUF conversion
# needs the fp16 HF model on disk (~14GB for the 7B model) AT THE SAME
# TIME as it writes the output file (~8GB minimum, even for the smallest
# practical single-pass Q8_0 format) -- 14+8=22GB will never fit in 19.5GB,
# regardless of cleanup/ordering tricks. The 3B model's fp16 size (~6GB)
# plus its Q8_0 output (~3.3GB) totals ~9.3GB, comfortably inside quota.
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"

# Dynamically select bfloat16 for modern hardware (Ampere+) or float16 for T4/P100
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Hardware-Optimized Compute Dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

print(f"Loading Tokenizer: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading Base Model: {MODEL_ID} with BitsAndBytes 4-bit NF4...")
# device_map="auto" would split the model's layers across both T4 GPUs
# (naive pipeline parallelism). That's fine for inference, but it breaks
# the Trainer's loss computation in Step 6 (transformers 4.46's
# num_items_in_batch normalization ends up on a different GPU than the
# model's final layers -> "Expected all tensors to be on the same
# device" RuntimeError). A 4-bit 7B model only needs ~5-6GB, so pin the
# whole model to a single GPU instead.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",
    trust_remote_code=True
)

print(f"✅ Model loaded into VRAM. Memory allocated: {torch.cuda.memory_allocated(0)/(1024**3):.2f} GB")


Verified Active Transformers Version: 4.46.3
Hardware-Optimized Compute Dtype: torch.bfloat16
Loading Tokenizer: Qwen/Qwen2.5-Coder-3B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model loaded into VRAM. Memory allocated: 1.91 GB


In [5]:
# Step 5: Construct Parameter-Efficient LoRA Synaptic Adapter
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Setting up PEFT LoRA adapter for synaptic weight adaptation...")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Setting up PEFT LoRA adapter for synaptic weight adaptation...
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [6]:
# Step 6: Train Sarthika Synaptic Weights with Supervised Reinforcement Formatting
from trl import SFTTrainer
from transformers import TrainingArguments

output_dir = "./sarthika_lora_adapter"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    num_train_epochs=3,
    logging_steps=1,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none"
)

# NOTE: single-GPU visibility is now enforced at the OS level in Step 1
# (CUDA_VISIBLE_DEVICES="0"), set before torch/CUDA initialized anywhere
# in this kernel. That's what actually prevents Trainer's legacy
# nn.DataParallel wrap (which broke on the scalar num_items_in_batch
# tensor). Patching training_args._n_gpu here directly is NOT reliable,
# because SFTTrainer converts TrainingArguments into its own internal
# SFTConfig during __init__, silently dropping non-field private
# attributes like a manually-set _n_gpu.

# NOTE: dataset (from Step 3) has a "messages" column (chat-format
# role/content dialogues), not a "text" column. SFTTrainer auto-detects
# "messages" and formats it using the tokenizer's chat template, so
# dataset_text_field must NOT be passed here.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=2048
)

print("INITIATING SYNAPTIC CONSOLIDATION (TRAINING LOOP)...")
train_result = trainer.train()

print(f"Training Complete! Loss: {train_result.training_loss:.4f}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Synaptic LoRA Adapter saved to: {output_dir}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


INITIATING SYNAPTIC CONSOLIDATION (TRAINING LOOP)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.383800
2,2.740600
3,2.227300


Training Complete! Loss: 2.1172
Synaptic LoRA Adapter saved to: ./sarthika_lora_adapter


In [7]:
# Step 7: Merge LoRA Weights into Base Model (Zero Downtime Reification)
import gc
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Clearing training cache and freeing GPU memory...")
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

merged_dir = "./sarthika_merged_hf"

# Merge on CPU, not GPU. A float16 7B model is ~14GB of weights alone --
# on this single-GPU-restricted session (see Step 1) that leaves almost
# no headroom on a 14.56GB T4, which is exactly what OOM'd. Merging is
# pure tensor arithmetic (no forward/backward passes), so it doesn't
# need GPU speed. Host RAM (checked in Step 1) has plenty of room for
# a 7B fp16 model, so do the whole merge there instead.
print(f"Reloading base model {MODEL_ID} in float16 on CPU for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)

print("Attaching trained LoRA adapter (CPU)...")
peft_model = PeftModel.from_pretrained(base_model, "./sarthika_lora_adapter", device_map="cpu")

print("Merging weights into single consolidated model (CPU)...")
merged_model = peft_model.merge_and_unload()

print(f"Saving merged float16 model to: {merged_dir}...")
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)

del merged_model
del base_model
del peft_model
gc.collect()
torch.cuda.empty_cache()
print("Consolidated float16 weights successfully saved!")

# Free disk space: the original ~15GB downloaded HF cache for MODEL_ID is
# no longer needed now that merged_dir has everything (weights + tokenizer)
# saved locally. Deleting it now avoids running out of disk space during
# GGUF conversion in Step 8, which needs room for its own multi-GB files.
import shutil
hf_cache = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.isdir(hf_cache):
    freed_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(hf_cache) for f in files
    ) / (1024**3)
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f"🧹 Cleared original HF download cache, freed ~{freed_gb:.2f} GB of disk space.")



Clearing training cache and freeing GPU memory...
Reloading base model Qwen/Qwen2.5-Coder-3B-Instruct in float16 on CPU for merging...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Attaching trained LoRA adapter (CPU)...
Merging weights into single consolidated model (CPU)...
Saving merged float16 model to: ./sarthika_merged_hf...
Consolidated float16 weights successfully saved!
🧹 Cleared original HF download cache, freed ~11.52 GB of disk space.


In [8]:
# Step 8: Convert Consolidated Weights into CPU-Optimized GGUF (Q4_K_M)
import subprocess
import os
import sys
import shutil

print("Executing llama.cpp GGUF conversion pipeline...")

merged_dir = "./sarthika_merged_hf"
q8_gguf = "./sarthika_q8_0.gguf"
final_q4_gguf = "./sarthika_agi_v1_q4_k_m.gguf"

def report_disk(label):
    total, used, free = shutil.disk_usage(".")
    print(f"[{label}] Disk free: {free / (1024**3):.2f} GB / {total / (1024**3):.2f} GB total")

# Clean up any partial/corrupt files left over from a previous failed run --
# a crashed conversion leaves a truncated, unusable GGUF sitting on disk,
# wasting space that's needed for the retry.
for stale in ["./sarthika_f16.gguf", q8_gguf, final_q4_gguf]:
    if os.path.exists(stale):
        os.remove(stale)
        print(f"🧹 Removed stale/partial file from a previous attempt: {stale}")

report_disk("before conversion")

# Convert directly to Q8_0 in a single pass (skip the intermediate F16
# GGUF entirely). A 7B model's F16 GGUF is ~15GB -- combined with the
# ~14GB merged_dir it reads from, that needs ~29GB of disk at once, which
# is more than this environment's quota (observed failing around ~19-20GB
# used). Q8_0 is about half the size (~7.6GB) and convert_hf_to_gguf.py
# writes it directly from the HF weights without ever materializing a
# full F16 copy, roughly halving the peak disk this step needs.
print("1. Converting Hugging Face model directly to Q8_0 GGUF (single pass, no F16 intermediate)...")
subprocess.run([
    sys.executable, "llama.cpp/convert_hf_to_gguf.py",
    merged_dir,
    "--outfile", q8_gguf,
    "--outtype", "q8_0"
], check=True)

# Free merged_dir (~14GB) immediately -- everything needed now lives in q8_gguf.
if os.path.exists(merged_dir):
    shutil.rmtree(merged_dir, ignore_errors=True)
    print("🧹 Removed merged_hf directory to free disk space.")
report_disk("after Q8_0 conversion")

# 2. Quantize Q8_0 -> Q4_K_M. llama-quantize can re-quantize from an
# already-quantized source (not just F16/F32), so this works fine and
# needs far less peak disk than quantizing from a full F16 file would.
print("2. Quantizing GGUF model to Q4_K_M (4-bit AVX2 optimized)...")
quant_candidates = [
    "llama.cpp/build/bin/llama-quantize",
    "llama.cpp/build/llama-quantize",
    "llama.cpp/llama-quantize"
]
quant_bin = next((c for c in quant_candidates if os.path.exists(c)), None)
if quant_bin is None:
    raise FileNotFoundError("llama-quantize binary not found! Please check llama.cpp build.")

# --allow-requantize is required here: llama-quantize refuses to
# "double quantize" (go from one already-quantized format to another,
# i.e. Q8_0 -> Q4_K_M) by default as a safety check, since it assumes
# you're normally starting from full-precision F16/F32. We're
# deliberately requantizing from Q8_0 to save disk space (see Step 8's
# comments above), so this needs to be explicitly allowed.
subprocess.run([
    quant_bin,
    "--allow-requantize",
    q8_gguf,
    final_q4_gguf,
    "Q4_K_M"
], check=True)

# 3. Clean up intermediate Q8_0 GGUF to free disk space
if os.path.exists(q8_gguf):
    os.remove(q8_gguf)
    print("Removed intermediate Q8_0 GGUF to conserve disk space.")

report_disk("after cleanup")

file_size_gb = os.path.getsize(final_q4_gguf) / (1024**3)
print("=" * 65)
print("SUCCESS: Sarthika AGI Core v1 Exported as GGUF!")
print(f"Target File : {final_q4_gguf}")
print(f"File Size   : {file_size_gb:.2f} GB")
print("Deployment  : CPU-Ready (Runs at 15-25 tokens/sec with 0 GPU!)")
print("=" * 65)


Executing llama.cpp GGUF conversion pipeline...
[before conversion] Disk free: 13.26 GB / 19.52 GB total
1. Converting Hugging Face model directly to Q8_0 GGUF (single pass, no F16 intermediate)...


INFO:hf-to-gguf:Loading model: sarthika_merged_hf
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {2048, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {11008, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.bias,   

🧹 Removed merged_hf directory to free disk space.
[after Q8_0 conversion] Disk free: 15.97 GB / 19.52 GB total
2. Quantizing GGUF model to Q4_K_M (4-bit AVX2 optimized)...


version: 0.5.0-dev (build 1, commit 9adc7f4)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing './sarthika_q8_0.gguf' to './sarthika_agi_v1_q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 32 key-value pairs and 434 tensors from ./sarthika_q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:            general.sampling.penalty_repeat f32              = 1.050000


llama_quantize: quantize time = 138790.54 ms
llama_quantize:    total time = 138790.54 ms
Removed intermediate Q8_0 GGUF to conserve disk space.
[after cleanup] Disk free: 17.23 GB / 19.52 GB total
SUCCESS: Sarthika AGI Core v1 Exported as GGUF!
Target File : ./sarthika_agi_v1_q4_k_m.gguf
File Size   : 1.80 GB
Deployment  : CPU-Ready (Runs at 15-25 tokens/sec with 0 GPU!)


In [9]:
# Step 9: Download Verification & Next Steps
import os

gguf_path = "./sarthika_agi_v1_q4_k_m.gguf"

if os.path.exists(gguf_path):
    print("=" * 65)
    print("YOUR PERSONAL AGI MODEL IS READY FOR DOWNLOAD!")
    print("=" * 65)
    size_mb = os.path.getsize(gguf_path) / (1024**2)
    print(f"File Name: {os.path.basename(gguf_path)}")
    print(f"File Size: {size_mb:.2f} MB (~3.8 GB)")
    print("""
HOW TO DOWNLOAD FROM KAGGLE:
1. In the right panel of your Kaggle notebook, look under 'Output' -> '/kaggle/working/'.
2. Find 'sarthika_agi_v1_q4_k_m.gguf'.
3. Click the three dots (...) next to it and select 'Download'.
4. Place the downloaded file into your local 'sarthika' project folder on your PC!

RUNNING ON YOUR PC (NO GPU REQUIRED):
Run the provided 'local_sarthika_engine.py' script on your PC:
   $ python3 local_sarthika_engine.py
It will load this GGUF file into CPU RAM and run lightning-fast queries!
""")
else:
    print("GGUF file not found. Please verify previous steps.")


YOUR PERSONAL AGI MODEL IS READY FOR DOWNLOAD!
File Name: sarthika_agi_v1_q4_k_m.gguf
File Size: 1840.50 MB (~3.8 GB)

HOW TO DOWNLOAD FROM KAGGLE:
1. In the right panel of your Kaggle notebook, look under 'Output' -> '/kaggle/working/'.
2. Find 'sarthika_agi_v1_q4_k_m.gguf'.
3. Click the three dots (...) next to it and select 'Download'.
4. Place the downloaded file into your local 'sarthika' project folder on your PC!

RUNNING ON YOUR PC (NO GPU REQUIRED):
Run the provided 'local_sarthika_engine.py' script on your PC:
   $ python3 local_sarthika_engine.py
It will load this GGUF file into CPU RAM and run lightning-fast queries!



In [11]:
import zipfile
from IPython.display import FileLink

# Zip the GGUF model
with zipfile.ZipFile('model_download.zip', 'w', zipfile.ZIP_STORED) as zipf:
    zipf.write('/kaggle/working/sarthika_agi_v1_q4_k_m.gguf', arcname='sarthika_agi_v1_q4_k_m.gguf')

# Generate link for the zip file
FileLink('model_download.zip')

/kaggle/working/model_download.zip